# OY1 — ARC-AGI-3 reproduction

Reproduce the configuration used for OY1's completed public run on 9 September 2026:
**100.0 score, 25/25 games and 183/183 levels** with GPT-6 Astra at high reasoning.

The notebook downloads the exact evaluated source from a pinned repository commit
and checks its hashes before execution. To use a local copy, set `BUNDLE_PATH`.
See the [reproduction guide](../docs/reproduction.md) for setup and source identity.

Use Linux x86_64, Python 3.12, CPU and internet access. The default `MODE="fixture"`
checks the installation without model calls or credentials. For a new benchmark
run, set `MODE="run"`, a spending cap and its authorization record, and attach
`OPENAI_API_KEY` and `ARC_API_KEY` through Kaggle Secrets. The reported $415.37
cost applies to the completed run; a new run may cost a different amount.

Copyright 2026 Orca Labs sp. z o.o. Apache-2.0; see the repository's license notices.


In [ ]:
# Reproduction wrapper by Orca Labs sp. z o.o.; Apache-2.0.
# Uses the exact source snapshot from the completed public run.
import hashlib
import io
import math
import pathlib
import platform
import sys
import tempfile
import time
import zipfile
import urllib.request

NOTEBOOK_STARTED = time.monotonic()
MODE = "fixture"  # "fixture" makes no API calls; "run" selects all 25 public games.
APPROVED_USD = None  # Set an evaluator-approved cap only for MODE="run".
APPROVAL_RECORD = ""  # Record the evaluator's authorization for this new run.
if MODE not in ("fixture", "run"):
    raise ValueError("Choose fixture or run explicitly")
if MODE == "run" and (
    isinstance(APPROVED_USD, bool)
    or not isinstance(APPROVED_USD, (int, float))
    or not math.isfinite(APPROVED_USD)
    or not 0 < APPROVED_USD <= 10000
    or not isinstance(APPROVAL_RECORD, str)
    or not APPROVAL_RECORD.strip()
):
    raise ValueError("A new evaluator-approved cap and approval record are required")
if sys.platform != "linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 12):
    raise ValueError("Select Linux x86_64 with Python 3.12")
# Use a local transport copy or download the pinned evaluated bundle.
BUNDLE_PATH = None  # Optional path to oy1-evaluated-source.zip or .bin.
BUNDLE_URL = "https://raw.githubusercontent.com/OYLabsAI/arc-agi-3-api-harness/4d7586ee1b7e5d5545ac7ca3fec7c3e6ef112d2f/assets/api9-source-and-licenses.zip"
BUNDLE_SHA256 = "be904f8b6e72cea0ca469d9bee1697f2fa5927fce8145a345546bc772f687f7d"
if BUNDLE_PATH is not None:
    with pathlib.Path(BUNDLE_PATH).expanduser().open("rb") as source:
        bundle_bytes = source.read(10_000_001)
else:
    with urllib.request.urlopen(BUNDLE_URL, timeout=60) as source:
        if not source.url.startswith("https://"):
            raise ValueError("Source redirect must remain HTTPS")
        bundle_bytes = source.read(10_000_001)
if len(bundle_bytes) > 10_000_000 or hashlib.sha256(bundle_bytes).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Licensed source bundle size/hash mismatch")
download_dir = pathlib.Path(tempfile.mkdtemp(prefix="oy1-evaluated-source-"))
(download_dir / "oy1-evaluated-source.zip").write_bytes(bundle_bytes)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as bundle:
    source_bytes = bundle.read("source.zip")
archive_sha = "53cc83d5680ff47633d00f2f4f6ace8d9b10e1484779a810faad5247bcd5d009"
if hashlib.sha256(source_bytes).hexdigest() != archive_sha:
    raise ValueError("Evaluated source archive hash mismatch")
archive = download_dir / "source.zip"
archive.write_bytes(source_bytes)
CONFIG = {
    "mode": MODE,
    "source_archive": str(archive),
    "archive_sha256": archive_sha,
    "lock_sha256": "c14a6280f5e222cd621b3784b99ff5853cb7fd5cd2632b8465dc56309c21a529",
    "release_manifest_sha256": "84a784778cf0a2b01d5f93380445b050b4bbee56fbb865d6e8bf4ebd9cde1053",
    "plan": "plans/pilot.json" if MODE == "fixture" else "plans/public-repeat.json",
    "max_elapsed_seconds": 3600 if MODE == "fixture" else 41400,
    "output_root": str(
        pathlib.Path(
            tempfile.mkdtemp(
                prefix="oy1-output-",
                dir="/kaggle/working" if pathlib.Path("/kaggle/working").is_dir() else None,
            )
        )
    ),
    "approved_usd": APPROVED_USD,
    "approval_record": APPROVAL_RECORD,
}
print({"mode": MODE, "python": sys.version.split()[0], "source_manifest": CONFIG["release_manifest_sha256"]})

In [ ]:
# Public wrapper modification: explicit fixture/run dispatch; paid observer/audit preserved.
import hashlib
import pathlib
import tempfile
import types
import urllib.request
import zipfile
from datetime import UTC

archive = str(CONFIG["source_archive"])
if archive.startswith("https://"):
    target = pathlib.Path(tempfile.mkdtemp(prefix="oy-source-")) / "source.zip"
    with urllib.request.urlopen(archive, timeout=60) as response, target.open("wb") as stream:
        if not response.url.startswith("https://"):
            raise ValueError("Source redirect must remain HTTPS")
        count = 0
        while chunk := response.read(65536):
            count += len(chunk)
            if count > 100_000_000 or time.monotonic() - NOTEBOOK_STARTED >= CONFIG["max_elapsed_seconds"]:
                raise ValueError("Source download exceeded the size/time budget")
            stream.write(chunk)
    archive = str(target)
if hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest() != CONFIG["archive_sha256"]:
    raise ValueError("Frozen source archive hash mismatch")
with zipfile.ZipFile(archive) as source:
    bootstrap_source = source.read("scripts/bootstrap.py")
bootstrap = types.ModuleType("oy_verified_bootstrap")
exec(compile(bootstrap_source, "verified-source/scripts/bootstrap.py", "exec"), bootstrap.__dict__)
CONFIG["source_archive"] = archive


def evaluator_secrets():
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    return {name: client.get_secret(name) for name in ("OPENAI_API_KEY", "ARC_API_KEY")}


if CONFIG["mode"] == "fixture":
    RESULT = bootstrap.launch(CONFIG, None, started=NOTEBOOK_STARTED)
    print({"scope": "synthetic fixture only; no benchmark score", **RESULT})
    if RESULT["exit_code"] != 0:
        raise RuntimeError("Fixture validation failed; inspect its retained output")
else:
    """Read-only notebook progress. Never imported by the evaluated solver."""
    import json
    import sqlite3
    import threading
    import time
    from datetime import datetime
    from pathlib import Path

    def progress_snapshot(run_directory, cap_usd, started):
        run = Path(run_directory)
        manifest = json.loads((run / "manifest.json").read_text())
        results_path = run / "results.json"
        results = json.loads(results_path.read_text()) if results_path.exists() else []
        selected = manifest["selected_games"]
        finished = {r["game_id"] for r in results}
        current_id = next(
            (g for g in selected if g not in finished and (run / g / "current.json").exists()), None
        )
        current = json.loads((run / current_id / "current.json").read_text()) if current_id else None
        db = sqlite3.connect(
            (run / "cost-ledger.sqlite").resolve().as_uri() + "?mode=ro", uri=True, timeout=0.05
        )
        try:
            db.execute("PRAGMA query_only=ON")
            amount, operations, unresolved = db.execute(
                "SELECT COALESCE(SUM(charge),0), COUNT(*), "
                "COALESCE(SUM(CASE WHEN status != 'accounted' THEN 1 ELSE 0 END),0) FROM requests"
            ).fetchone()
        finally:
            db.close()
        snapshot = {
            "kind": "oy_progress",
            "at": datetime.now(UTC).isoformat(),
            "run_id": run.name,
            "games_selected": len(selected),
            "games_finished": len(results),
            "games_won": sum(r["won"] is True for r in results),
            "current_game": current_id,
            "current_levels_completed": current.get("levels_completed") if current else None,
            "current_levels_total": current.get("win_levels") if current else None,
            "actions_submitted": sum(r["actions_submitted"] for r in results)
            + (current.get("index", 0) if current else 0),
            "levels_completed": sum(r["levels_completed"] for r in results)
            + (current.get("levels_completed", 0) if current else 0),
            "charged_or_reserved_usd": amount / 1e6,
            "pending_or_uncertain_requests": unresolved,
            "dispatched_operations": operations,
            "run_cap_usd": cap_usd,
            "remaining_unreserved_usd": max(0, cap_usd - amount / 1e6),
            "largest_next_request_reserve_usd": 24.55,
            "elapsed_notebook_seconds": round(time.monotonic() - started, 1),
            "provisional": True,
        }
        terminal = run / "summary.json"
        if terminal.exists():
            summary = json.loads(terminal.read_text())
            snapshot["finalization_status"] = summary.get("finalization_status")
            snapshot["evaluation_error"] = summary.get("evaluation_error")
            snapshot["selected_set_score_percent"] = summary.get("selected_set_score_percent")
        return snapshot

    class ProgressObserver:
        def __init__(self, root, cap_usd, started, interval=60, emit=print):
            self.root = Path(root)
            self.cap_usd, self.started, self.interval, self.emit = cap_usd, started, interval, emit
            self.stopped = threading.Event()
            self.thread = threading.Thread(target=self._loop, name="oy-read-only-progress", daemon=True)

        def sample(self):
            try:
                manifests = list(self.root.glob("*/runs/*/manifest.json"))
                if not manifests:
                    value = {
                        "kind": "oy_progress",
                        "phase": "setup",
                        "at": datetime.now(UTC).isoformat(),
                    }
                elif len(manifests) != 1:
                    value = {"kind": "oy_progress_error", "reason": "ambiguous_run_identity"}
                else:
                    value = progress_snapshot(manifests[0].parent, self.cap_usd, self.started)
            except Exception as exc:
                # Concurrent JSON writes and short-lived DB locks can prevent a snapshot.
                # Do not print exception text, raw database rows or model content.
                value = {"kind": "oy_progress_unavailable", "error_type": type(exc).__name__}
            self.emit(json.dumps(value, sort_keys=True), flush=True)

        def _loop(self):
            while not self.stopped.is_set():
                self.sample()
                self.stopped.wait(self.interval)

        def start(self):
            self.thread.start()

        def stop(self):
            self.stopped.set()
            self.thread.join(timeout=2)
            self.sample()

    if list(pathlib.Path(CONFIG["output_root"]).glob("*/runs/*/manifest.json")):
        raise RuntimeError("This output root already contains a run; refuse a silent repeat")
    observer = ProgressObserver(CONFIG["output_root"], CONFIG["approved_usd"], NOTEBOOK_STARTED)
    observer.start()
    try:
        RESULT = bootstrap.launch(
            CONFIG, evaluator_secrets if CONFIG["mode"] == "run" else None, started=NOTEBOOK_STARTED
        )
    finally:
        observer.stop()
    print(RESULT)

    import subprocess

    work = pathlib.Path(RESULT["work_directory"])
    run_dirs = sorted((work / "runs").glob("*/evidence-manifest.json"))
    if len(run_dirs) != 1:
        raise ValueError("Exactly one fresh run must be present")
    run_dir = run_dirs[0].parent
    evidence_sha = hashlib.sha256(run_dirs[0].read_bytes()).hexdigest()
    audit_command = [
        sys.executable,
        "-I",
        "-S",
        "-B",
        str(work / "source/scripts/independent_audit.py"),
        str(run_dir),
        "--expected-release-sha256",
        CONFIG["release_manifest_sha256"],
        "--expected-evidence-sha256",
        evidence_sha,
    ]
    if RESULT["exit_code"] != 0:
        audit_command.append("--allow-incomplete")
    remaining = CONFIG["max_elapsed_seconds"] - (time.monotonic() - NOTEBOOK_STARTED)
    if remaining <= 0:
        raise TimeoutError("No time remains for independent audit")
    with (work / "independent-audit.json").open("w") as report:
        audit_code = subprocess.run(
            audit_command, stdout=report, stderr=subprocess.STDOUT, timeout=min(120, remaining), check=False
        ).returncode
    summary = json.loads((run_dir / "summary.json").read_text())
    card = json.loads((run_dir / "scorecard.json").read_text()).get("sdk_scorecard") or {}
    receipt = {
        "source_manifest_sha256": CONFIG["release_manifest_sha256"],
        "evidence_manifest_sha256": evidence_sha,
        "run_id": run_dir.name,
        "independent_audit_exit_code": audit_code,
        "elapsed_including_audit_seconds": time.monotonic() - NOTEBOOK_STARTED,
        "raw_public_score": card.get("score"),
        "public_100_status": summary.get("public_100_status"),
        "games_selected": summary.get("games_selected"),
        "games_won": summary.get("games_won"),
        "cost": summary.get("cost"),
        "evaluation_scope": "public_benchmark_reproduction",
    }
    (work / "full-public-execution.json").write_text(json.dumps(receipt, indent=2) + "\n")
    print(receipt)
    if audit_code:
        raise RuntimeError("Independent evidence audit failed")

    if RESULT["exit_code"] != 0:
        raise RuntimeError(
            "Evaluation failed or stopped. Preserve this attempt and inspect its private artifacts."
        )